# Building a crossword grid

A walkthrough of one puzzle, from a list of words to a finished grid, stopping
at each stage to look at what the builder actually has in its hands.

The example is a British cryptic: a 15×15 blocked grid, about half the letters
unchecked, built around a theme. The same machinery makes American and barred
puzzles by changing a rule set and a library, and those get their own
notebooks.

**What we are going to do**

1. Read a dictionary and turn it into something a search can use
2. Take a grid pattern from a library of published ones
3. Ask which patterns could hold the theme at all
4. Seat the theme words
5. Fill everything else
6. Read the result back as a puzzle

Run the cells in order.

## Setup

The notebook lives in `notebooks/`, so first point Python at the repository root.

In [1]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from notebooks.draw import show, cells_of, clue_lists

print("working from", os.getcwd())

working from /Users/angus/Documents/Crosswords/Crossword Builder


## 1. The words

`load` reads a word list and normalises it. Each entry keeps two forms: `text`,
which is what goes in the grid — lowercase, no punctuation — and `surface`,
which is what a solver would see written down. That distinction matters later:
the grid holds `twelfthnight`, but the enumeration a setter prints is (7,5).

The flags are filters. `proper` marks capitalised entries and `phrase` marks
anything whose punctuation was stripped, which is how `it'll` becomes `itll`.

In [2]:
from crossword.words import load

entries = load("crossword/UKACD.txt")
print(f"{len(entries):,} entries")

for entry in entries[:3] + [e for e in entries if e.phrase][:2]:
    print(f"  text={entry.text!r:20} surface={entry.surface!r:20} "
          f"proper={entry.proper} phrase={entry.phrase}")

221,835 entries
  text='aardvark'           surface='aardvark'           proper=False phrase=False
  text='aardvarks'          surface='aardvarks'          proper=False phrase=False
  text='aardwolf'           surface='aardwolf'           proper=False phrase=False
  text='abadegg'            surface='a bad egg'          proper=False phrase=True
  text='abadhat'            surface='a bad hat'          proper=False phrase=True


### Turning them into a search structure

The filler asks one question over and over: *given this pattern of known and
unknown letters, which words fit?* Doing that with string comparison is far too
slow, so the index answers it with bitwise arithmetic instead.

For every length, every position and every letter, the index stores a bitset:
one bit per word, set if that word has that letter there. A pattern is then an
AND of one bitset per known letter, and the number of candidates is a popcount.

In [3]:
from crossword.index import Index

index = Index(entries)
bucket = index.lengths[5]
print(f"{len(bucket.words):,} five-letter words")

# Words matching  S _ E _ L
mask = bucket.match("s.e.l")
print(f"\nS_E_L matches {bucket.count('s.e.l')} words:")
print("  ", ", ".join(bucket.iterate(mask)))

10,162 five-letter words

S_E_L matches 16 words:
   sheal, sheel, shell, sheol, skell, smell, snell, speal, speel, spell, steal, steel, steil, stell, sweal, swell


## 2. The theme

The point of the project: fit as many words from a chosen list into one grid as
possible. Here is the theme.

In [4]:
theme = ["kestrel", "redwing", "wheatear", "fieldfare",
         "goldcrest", "nuthatch", "siskin", "dunnock"]

from collections import Counter
print("theme words by length:", sorted(Counter(len(w) for w in theme).items()))

theme words by length: [(6, 1), (7, 3), (8, 2), (9, 2)]


## 3. The grid library

Grids are not invented. They are taken from a library extracted from published
puzzles — 120 patterns, geometry only, drawn from 8,348 Guardian cryptics.

This matters more than it sounds. Randomly generated patterns that pass every
rule turn out to fill very badly; published ones fill immediately. A setter
knows things about where blocks go that no rule in this project captures, and
using their grids is how those things are inherited for free.

In [5]:
from crossword import library

patterns = library.load()
print(f"{len(patterns)} patterns, most-used first\n")

first = patterns[0]
print(f"used by {first.uses} published puzzles")
show(first.grid())

120 patterns, most-used first

used by 458 published puzzles


1,,2,,3,,4,,5,,6,,7,,8
,,,,,,,,,,,,,,
9,,,,,,10,,,,,,,,
,,,,,,,,,,,,,,
11,,,,,,,,,,,12,,,
,,,,,,,,,,13,,,,
,,,,14,,15,,,,,,,,
16,,17,,,,,,,,,,,,
18,,,,,,,,,,,,,,
,,,,,,,,,,,,19,,20
21,,,,,22,,,23,,,,,,


### What the builder sees in a pattern

Not squares — *slots*. Every maximal run of white cells three or more long is
an entry, and the cells where an across entry crosses a down entry are
*checked*: the solver gets two chances at them. Everything the rules say is
said about slots and checked cells, never about blocks directly.

In [6]:
grid = first.grid()
slots = grid.slots()
checked = grid.checked_cells()

print(f"{len(slots)} entries, {len(checked)} of {grid.size**2} cells checked")
print("lengths:", sorted(Counter(s.length for s in slots).items()))
print()
print("C = checked, U = unchecked, # = block")
print(grid.annotate(gap=" "))

30 entries, 58 of 225 cells checked
lengths: [(4, 4), (5, 4), (6, 4), (7, 4), (8, 4), (9, 4), (10, 4), (11, 2)]

C = checked, U = unchecked, # = block
C U C U C U C # C U C U C U C
U # U # U # U # U # U # U # U
C U C U C # C U C U C U C U C
U # U # U # U # U # U # U # U
C U C U C U C U C U # U C U C
U # U # U # # # U # U # U # U
# # # # C U C U C U C U C U C
U # U # U # U # U # U # U # U
C U C U C U C U C U C # # # #
U # U # U # U # # # U # U # U
C U C U # U C U C U C U C U C
U # U # U # U # U # U # U # U
C U C U C U C U C # C U C U C
U # U # U # U # U # U # U # U
C U C U C U C # C U C U C U C


### The rules

Six predicates decide whether a pattern is one a setter would print. Each was
checked against real puzzles rather than argued from first principles — the
history of getting that wrong is in `BASELINE.md`.

In [7]:
from crossword.rules import validate, RuleSet

print("published pattern:", validate(grid) or "clean")

# A deliberately broken one: block a cell without its rotational partner.
broken = first.grid()
broken.blocks.add((0, 4))
broken._stamp += 1
broken._derived.clear()
for problem in validate(broken)[:4]:
    print(" ", problem)

published pattern: clean
  Violation(rule='run_length', detail='across at (0,5) length 2 is shorter than 3')
  Violation(rule='isolated_cell', detail='cell (0, 5) is in no entry')
  Violation(rule='consecutive_unchecked', detail='down at (0,6) length 5 has 2 unchecked cells in a row')
  Violation(rule='symmetry', detail='block (0, 4) has no partner at (14, 10)')


## 4. Which grids could hold the theme?

Before any search runs, an upper bound is free. A grid offering four seven-letter
entries cannot hold five seven-letter theme words, whatever the search does.
Matching theme lengths against the grid's supply gives a *ceiling*, and the
patterns are tried in that order.

In [8]:
from crossword import coverage

scored = []
for pattern in patterns:
    profile = pattern.profile()
    scored.append((coverage.ceiling(profile, theme), pattern))
scored.sort(key=lambda pair: -pair[0])

print(f"best ceiling in the library: {scored[0][0]} of {len(theme)} theme words")
print(f"how many patterns reach it:  "
      f"{sum(1 for c, _ in scored if c == scored[0][0])}")
print("\nlengths this pattern offers:",
      sorted(scored[0][1].profile().items()))

best ceiling in the library: 8 of 8 theme words
how many patterns reach it:  52

lengths this pattern offers: [(4, 4), (5, 4), (6, 4), (7, 4), (8, 4), (9, 4), (10, 4), (11, 2)]


## 5. Seating the theme, and filling the rest

Now the search. `best_over_library` walks the promising patterns, and for each
one tries to seat the theme words and then fill every remaining entry from the
dictionary.

Two searches are nested here, and they are quite different. Seating is a
branch-and-bound over which theme word goes in which slot. Filling is a
backtracking search with forward checking: at every node it computes the
candidate set for each empty slot, which serves both as the check that no slot
has been killed and as the ordering — expand the most constrained first.

In [9]:
import time

began = time.time()
result = coverage.best_over_library(
    patterns, index, theme, RuleSet(),
    time_limit=90, commonness=3.0, aim=0.85, seed=1,
)
print(f"{time.time() - began:.1f}s")
print(f"complete grid: {result.ok}")
print(f"theme words seated: {result.n} of {len(theme)} (ceiling was {result.ceiling})")
print("seated:", ", ".join(result.placed))

0.1s
complete grid: True
theme words seated: 8 of 8 (ceiling was 8)
seated: fieldfare, goldcrest, nuthatch, wheatear, dunnock, kestrel, redwing, siskin


### The finished grid

The theme words are tinted.

In [10]:
finished = result.grid
marked = [cell for word in result.placed for cell in cells_of(finished, word)]
show(finished, highlight=marked,
     caption=f"{len(finished.slots())} entries, "
             f"{len(result.placed)} of {len(theme)} theme words")

,1R,U,2N,I,3T,S,4E,L,5F,,6K,A,7G,O
,O,,O,,O,,C,,I,,E,,O,
8N,U,T,H,A,T,C,H,,9E,N,S,I,L,E
,E,,O,,E,,O,,L,,T,,D,
10I,S,O,P,O,D,,11I,N,D,I,R,E,C,T
,,,E,,,,S,,F,,E,,R,
12E,13S,T,R,U,14S,,15M,E,A,S,L,I,E,R
,C,,,,W,,,,R,,,,S,
16W,H,E,17A,T,E,A,18R,,19E,N,20D,I,T,E
,I,,L,,E,,E,,,,U,,,
21C,L,O,S,E,T,E,D,,22L,I,N,G,23U,A


## 6. Reading it back as a puzzle

A cell earns a number when an entry starts there in *either* direction, and a
cell starting both shares one number between them. That sharing is why the
across and down lists have gaps in them rather than each running 1, 2, 3.

In [11]:
across, down = clue_lists(finished)

print("ACROSS")
for number, answer in across[:8]:
    print(f"  {number:>3}  {answer.upper()}")
print("   ...")
print("DOWN")
for number, answer in down[:8]:
    print(f"  {number:>3}  {answer.upper()}")

ACROSS
    1  RUNITSELF
    6  KAGO
    8  NUTHATCH
    9  ENSILE
   10  ISOPOD
   11  INDIRECT
   12  ESTRUS
   15  MEASLIER
   ...
DOWN
    1  ROUES
    2  NOHOPER
    3  TOTED
    4  ECHOISM
    5  FIELDFARE
    6  KESTREL
    7  GOLDCREST
   13  SCHILLING


### Out to a file

`to_ipuz` and `to_exolve` write the grid in the two formats setters actually
use — ipuz for Exet and most desktop software, Exolve for a self-contained web
page. Enumerations come from the `surface` forms, which is why the distinction
at the very top mattered.

In [12]:
from crossword import export

surfaces = {e.text: e.surface for e in entries}
document = export.to_ipuz(finished, title="Dawn Chorus", author="",
                          surfaces=surfaces)
print("ipuz keys:", ", ".join(sorted(document)[:8]), "...")
print("dimensions:", document["dimensions"])
print("first three across clues:")
for clue in document["clues"]["Across"][:3]:
    print("  ", clue)

ipuz keys: author, clues, dimensions, kind, puzzle, showenumerations, solution, title ...
dimensions: {'width': 15, 'height': 15}
first three across clues:
   {'number': '1', 'label': '1', 'clue': '(3,6)', 'answer': 'RUN ITSELF'}
   {'number': '6', 'label': '6', 'clue': '(4)', 'answer': 'KAGO'}
   {'number': '8', 'label': '8', 'clue': '(8)', 'answer': 'NUTHATCH'}


## Where this goes next

This is the shape of the process. Each stage has considerably more inside it,
and the plan is a notebook for each:

- **the dictionary and the index** — bitsets, why one universe per length
- **the rules** — how each was derived from published puzzles, and the one that
  was wrong for months
- **choosing a grid** — length profiles, ceilings, and why grid choice decides
  more than the fill search does
- **the fill search** — MRV, forward checking, restarts, and how word
  familiarity is scored so the grid does not fill with ISLE and OVER
- **the other styles** — American and barred, and what changed for each
- **ninas and pangrams** — constraints the setter adds on top